# Indexing of a map guided by the UB matrices of the neighbouring points

Indexing from scratch (angular LUT) takes minutes per image for a large cell (e.g. monoclinic NiMnGa); checking and
refining a UB matrix that already indexes a neighbouring point takes 1-3 s (see `LaueTools/indexing_neighbours.py`).

**Steps**
1. **Seeds**: the points of a regular grid of the map (rectangular or hexagonal) are indexed from scratch
   (`configs/indexing_seeds.yaml`, .fit files in `seeds_<material>`), and each solution found is checked with the
   **criteria of a correct solution** `SEED_CRITERIA`: from its UB matrix, steps [e_max, tolerance angle, min nb of
   matches] in sequence (links, refinement, nb of indexed spots >= min), e.g. for NiMnGa 0.2 deg at 18 keV, then
   up to 27 keV at 0.2 deg and 0.1 deg at the end, always more than 100 indexed spots, and a mean pixel deviation
   below `SEED_MAX_PIXDEV` (1 pixel by default) at the end. A few minutes on one node.
2. **Validation of the seeds** by you: table (nb of spots at each step of the criteria, pass / FAIL), map,
   orientation groups, optional thresholds, seeds excluded or included by hand. By default only the seeds passing
   the criteria are validated. The validated seeds are written in `configs/validated_seeds.json`.
3. **Map**: all the other points, each one checking first the UB matrix of the previous point of its process, the UB
   matrices of the points already indexed within `radius` (validated seeds included), the library of the
   orientations (validated seeds and points indexed from scratch), then the optional reference UB matrices; the
   first one giving `min_nb_matches` indexed spots (and `min_matching_rate`) is accepted, otherwise the point is
   indexed from scratch. The validated seeds are refined again with the parameters of the map (all the .fit files
   of the map refined the same way). Job array on `magnifix3` (3 nodes sharing the UB matrices through
   `<fit_folder>/ub_store`).

**Running again** (other parameters, or `REDO = True`): nothing to delete or rename. The previous results folder is
renamed `<folder>_previous_<date>` (delete it when you do not need it) and the step is computed again. Results
computed with the same parameters are reloaded. A SLURM job cancelled (or stopped by its time limit) is detected
and the step is computed again; its partial results are shown by the results cell (`IN.load_store_results`).

## Settings

In [ ]:
# indexing configuration of the map (peak search done: .cor files in cor_folder)
CONFIG = '/data/visitor/<proposal>/bm32/<date>/PROCESSED_DATA/<sample>/<dataset>/scan0001/configs/indexing.yaml'

# changes of indexing parameters of CONFIG (as in the quickstart notebook), e.g. the refinement tolerances (deg).
# The best parameters of the indexing from scratch depend on the microstructure: check that most seeds pass the
# criteria (step 2). E.g. for <sample> (NiMnGa), image 8318: 'central spots indices' 'zones', nlutmax 4,
# AngleTolLUT and MATCHINGRATE_ANGLE_TOL 0.3, stop_Nb_Matches 115 and refinement_steps from 18 to 27 keV give the
# correct solution (245 spots, 0.57 pixel at 27 keV) in 11 s, whereas 'to5', nlutmax 6, 0.2 deg and stop_Nb_Matches 40
# give a wrong one (54 spots at 18 keV)
INDEXING = {}

# step 1: grid of the seeds (indexed from scratch). Spacing: seed_spacing (micron, the same in real space along
# both motors: e.g. 500 micron = 15 points along yps (33 micron steps), 47 along xps (10.6 micron steps)),
# or, if seed_spacing is None, seed_step map points along both motors
SEEDS = {'seed_grid': 'rectangular',  # or 'hexagonal' (most compact)
         'seed_spacing': 350,         # micron (map step sizes); None: seed_step is used
         'seed_step': 20}              # map points, only if seed_spacing is None
# criteria of a correct solution (seeds, reference UB matrices): steps [e_max (keV), tolerance angle (deg), min nb of
# indexed spots], done in sequence from the UB matrix found (links, refinement); None: no check
SEED_CRITERIA = [[18, 0.2, 100], [21, 0.2, 100], [24, 0.2, 100], [27, 0.2, 100], [27, 0.1, 100]]
SEED_MAX_PIXDEV = 1.0   # pixel, max mean pixel deviation of the last step of SEED_CRITERIA (None: not checked)

# step 2: validation of the seeds
SEED_VALIDATION = {'require_criteria': True,   # SEED_CRITERIA passed
                   'min_nb_spots': 0,          # and optional thresholds: nb of indexed spots (indexing)
                   'min_matching_rate': 0,     # %
                   'orientations': None,       # orientation groups kept (numbers of the table), None: all
                   'exclude': [],              # image indices of seeds rejected by hand
                   'include': []}              # image indices of seeds validated by hand

# step 3: map (written in run: neighbours of CONFIG)
NEIGHBOURS = {'radius': 10,            # map points (square neighbourhood) where indexed UB matrices are taken
              'min_nb_matches': 50,    # nb of indexed spots to accept a UB matrix guess
              'min_matching_rate': 6,  # and matching rate (%)
              # 'front': points next to the most accepted points first, one task per point, indexing from scratch
              # deferred to the end; 'tiles': tiles of the map
              'dispatch': 'front',
              # check of a raw (not refined) candidate UB: tolerance angle (deg) of the spots matching and
              # matching rate (%) needed to refine it (None: MATCHINGRATE_ANGLE_TOL and MinimumMatchingRate
              # of dict_indexrefine, as for the indexing from scratch). Then refinement as for the indexing
              # from scratch, and acceptance with min_nb_matches (and min_matching_rate) at its last step
              'check_angle_tol': None,
              'check_min_matching_rate': None}

# optional reference UB matrices checked by all points: distinct orientations of a previous indexation of the map
# (its indexing.yaml), images with at least REFERENCE_MIN_SPOTS indexed spots; None: no reference
REFERENCE_FROM = None
REFERENCE_MIN_SPOTS = 120
NB_REFERENCES = 6

# computer
EXECUTION = 'slurm'              # 'slurm': jobs on the cluster, 'local': this machine, 'auto'
SEEDS_MACHINE = 'magnifix'       # step 1 (~100 seeds): 1 node
SLURM_MACHINE = 'magnifix3'      # step 3: 3 nodes of 192 cpus (job array of 3 jobs, 3 bands of the map)
NB_CPUS = None                   # per job (None: all cpus of a node)
TIME_LIMIT_SEEDS = '01:00:00'    # hh:mm:ss
TIME_LIMIT = '02:00:00'
REDO = False                     # True: compute again (same parameters)

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "1"
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import LaueTools.map_workflow as MW
import LaueTools.indexing_batch as IB
import LaueTools.indexing_neighbours as IN
import LaueTools.generaltools as GT

execution = MW.environment(EXECUTION)
icfg = IB.load_config(CONFIG)
if INDEXING:
    icfg = IB.save_config(icfg, CONFIG, overwrite=True, **INDEXING)
assert IB.check_config(icfg), 'fix the configuration'

## Step 1: seeds indexed from scratch

In [ ]:
scfg = IN.seeds_config(icfg, SEEDS, criteria=SEED_CRITERIA, max_pixdev=SEED_MAX_PIXDEV)   # configs/indexing_seeds.yaml

# seeds in real space, over the distance of each map point to its nearest seed: check the grid before indexing
fig, ax = plt.subplots(figsize=(8, 8))
IN.plot_seed_grid(scfg, ax=ax)
plt.show()


In [ ]:
seeds = MW.run_step('indexing', scfg, execution, nb_cpus=NB_CPUS, machine=SEEDS_MACHINE,
                    time_limit=TIME_LIMIT_SEEDS, redo=REDO)

## Step 2: validation of the seeds

Criteria: nb of indexed spots at each step of `SEED_CRITERIA` (`-`: step not reached), pass / FAIL. Orientation
groups: seeds with the same orientation (symmetry equivalent UB matrices). A seed alone in its orientation group may
still be a wrong solution: exclude it (`exclude`), or change `SEED_VALIDATION` and run this cell again. If most
seeds fail the criteria, change the indexing parameters (`INDEXING`) and run step 1 again.

In [ ]:
table = IN.seed_table(seeds, scfg)
validated = IN.validate_seeds(table, **SEED_VALIDATION)
IN.print_seed_table(table, validated)
print(f'{len(validated)} validated seeds (*)')
fig, ax = plt.subplots(figsize=(8, 5.5))
IN.plot_seeds(table, scfg, validated, ax=ax)
plt.show()

In [ ]:
# validated seeds -> configs/validated_seeds.json (used by step 3)
assert validated, 'no validated seed: change SEED_VALIDATION, or INDEXING and run step 1 again'
seeds_file = IN.save_validated_seeds(table, validated, scfg)

## Step 3: map

`usepreviousUB: neighbours`, `NEIGHBOURS`, `SEEDS`, the validated seeds and the reference UB matrices are written in CONFIG.

In [ ]:
refs = []
if REFERENCE_FROM:
    refs = IN.distinct_orientations(IB.load_config(REFERENCE_FROM), min_nb_spots=REFERENCE_MIN_SPOTS,
                                    nmax=NB_REFERENCES)
icfg = IB.save_config(icfg, CONFIG, overwrite=True, usepreviousUB='neighbours', seeds_file=str(seeds_file),
                      ub_matrices={**icfg['ub_matrices'], **{r['name']: r['UB'] for r in refs}},
                      reference_UBs=[r['name'] for r in refs], **SEEDS, **NEIGHBOURS)
assert IB.check_config(icfg), 'fix the configuration'
params = IB.params_from_config(icfg)
listfiles = IB.select_corfiles(icfg)
print(IN.check_description(params))
if refs:   # criteria (or nb of indexed spots) of each reference on a few images (~2-5 s per image and matrix)
    checks = IN.check_references(listfiles, params, nb_images=6, criteria=SEED_CRITERIA, max_pixdev=SEED_MAX_PIXDEV)

The notebook waits for the end of the job (logs in `<scan folder>/slurm_jobs/<job>/logs`). If the wait is interrupted, running the cell again waits for the same job. **Job cancelled** (or time limit reached): the results cell shows the images already treated; running this cell again indexes the map again.

In [ ]:
fits = MW.run_step('indexing', icfg, execution, nb_cpus=NB_CPUS, machine=SLURM_MACHINE, time_limit=TIME_LIMIT,
                   redo=REDO)

## Results

In [ ]:
# job interrupted, cancelled or time limit reached (no results file): results of the images already
# treated, read from the UB store (their .fit files are in fit_folder)
if globals().get('fits') is None:
    fits = IN.load_store_results(icfg)

if fits is not None:
    IN.guide_statistics(fits)
    summary = IB.summarize_results(fits, icfg['scan']['mapdims'])
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    MW.plot_map(summary['Nbindexedimages2D'], icfg, 'grain 0: nb of indexed spots', ax=axes[0])
    MW.plot_map(summary['MRindexedimages2D'], icfg, 'grain 0: matching rate (%)', ax=axes[1])
    IN.plot_sources(fits, icfg, ax=axes[2])
    plt.tight_layout()
    plt.show()

In [ ]:
grains = MW.load_grains(icfg)
fig, mask = MW.quality_maps(grains, icfg)
plt.show()
fig = MW.orientation_maps(grains, icfg, mask=mask)
plt.show()

In [ ]:
fig = MW.pole_figures(grains, icfg, mask=mask)
plt.show()